<h1>Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ( accuracy_score, precision_score,recall_score,f1_score,roc_auc_score, confusion_matrix)

sns.set_theme()


<h1>Data Collection

In [ ]:
url = "https://cdn.theforage.com/vinternships/companyassets/Zbnc2o4ok6kD2NEXx/2kCX23cgKgCumeEam/1721851467492/Customer_Churn_Data_Large.xlsx"

excel = pd.ExcelFile(url)

In [ ]:
print(excel.sheet_names)

In [ ]:
df1 = pd.read_excel(url, sheet_name=0)
df2 = pd.read_excel(url, sheet_name=1)
df3 = pd.read_excel(url, sheet_name=2)
df4 = pd.read_excel(url, sheet_name=3)
df5 = pd.read_excel(url, sheet_name=4)

<h1>Data PreProcessing

<h2>DF1

In [ ]:
df1.isnull().sum()

In [ ]:
df1.duplicated().sum()

In [ ]:
new_df1 = df1.copy()
new_df1['Gender'] = new_df1['Gender'].map({'M':0, 'F':1})
new_df1['IncomeLevel'] =  new_df1['IncomeLevel'].map({'Low':0, 'Medium':1, 'High':2})
new_df1 = pd.get_dummies(new_df1, columns=['MaritalStatus'],drop_first=True, dtype=int)

new_df1.head()

In [ ]:
new_df1.describe()

<h2>DF2

In [ ]:
df2.isnull().sum()

In [ ]:
df2.duplicated().sum()

In [ ]:
new_df2 = df2.copy()
new_df2 = pd.get_dummies(new_df2, columns=['ProductCategory'],drop_first=True, dtype=int)

<h4>Standardization

In [ ]:
scaler_am = StandardScaler()
new_df2['AmountSpent'] = scaler_am.fit_transform(new_df2[['AmountSpent']])

In [ ]:
new_df2.head()

<h2>DF3

In [ ]:
df3.isnull().sum()

In [ ]:
df3.duplicated().sum()

In [ ]:
new_df3 = df3.copy()
new_df3 = pd.get_dummies(new_df3, columns=['InteractionType'],drop_first=True, dtype=int)
new_df3 = pd.get_dummies(new_df3, columns=['ResolutionStatus'],drop_first=True, dtype=int)

new_df3.head()

<h2>DF4

In [ ]:
new_df4 = df4.copy()
new_df4 = pd.get_dummies(new_df4, columns=['ServiceUsage'],drop_first=True, dtype=int)

<h4> Standardization

In [ ]:
scaler_lf = StandardScaler()
new_df4['LoginFrequency'] = scaler_lf.fit_transform(new_df4[['LoginFrequency']])

In [ ]:
new_df4.head()

<h2>DF5

In [ ]:
merged_df = df1.merge(df5, on='CustomerID')

merged_df.head()

In [ ]:
transaction_summary = df2.groupby('CustomerID')['AmountSpent'].agg(
    ['sum', 'count', 'mean']
).reset_index()
transaction_summary.columns = [
    'CustomerID',
    'TotalSpent',
    'TransactionCount',
    'AvgSpent'
]
interaction_counts = (
    df3.groupby('CustomerID')
    .size()
    .reset_index(name='InteractionCount')
)

In [ ]:
transaction_summary.head()

In [ ]:
interaction_counts = df3.groupby('CustomerID').size()
interaction_counts.head()

In [ ]:
merged_df.head()

In [ ]:
# Complaint count

complaint_count = (
    df3[df3['InteractionType'] == 'Complaint']
    .groupby('CustomerID')
    .size()
    .reset_index(name='ComplaintCount')
)

In [ ]:

# Unresolved count

unresolved_count = (
    df3[df3['ResolutionStatus'] == 'Unresolved']
    .groupby('CustomerID')
    .size()
    .reset_index(name='UnresolvedCount')
)

In [ ]:

# Merge transaction summary

merged_df = merged_df.merge(
    transaction_summary,
    on='CustomerID',
    how='left'
)

In [ ]:

# Merge interaction counts

merged_df = merged_df.merge(
    interaction_counts.reset_index(name='InteractionCount'),
    on='CustomerID',
    how='left'
)

In [ ]:

# Merge complaint counts

merged_df = merged_df.merge(
    complaint_count,
    on='CustomerID',
    how='left'
)

In [ ]:

# Merge unresolved counts

merged_df = merged_df.merge(
    unresolved_count,
    on='CustomerID',
    how='left'
)

In [ ]:

# Merge login/service dataset

merged_df = merged_df.merge(
    df4,
    on='CustomerID',
    how='left'
)

In [ ]:

# Fill missing values

merged_df = merged_df.fillna(0)

In [ ]:

# Inspect final merged dataset

merged_df.head()

merged_df.info()

merged_df.describe()

In [ ]:
model_df = merged_df.copy()

In [ ]:
# Gender encoding
model_df['Gender'] = model_df['Gender'].map({
    'M': 0,
    'F': 1
})

# Income level encoding
model_df['IncomeLevel'] = model_df['IncomeLevel'].map({
    'Low': 0,
    'Medium': 1,
    'High': 2
})

In [ ]:
model_df = pd.get_dummies(
    model_df,
    columns=[
        'MaritalStatus',
        'ServiceUsage'
    ],
    drop_first=True,
    dtype=int
)

model_df.head()

In [ ]:
scaler = StandardScaler()

numerical_cols = [
    'Age',
    'TotalSpent',
    'AvgSpent',
    'TransactionCount',
    'InteractionCount',
    'ComplaintCount',
    'UnresolvedCount',
    'LoginFrequency'
]

model_df[numerical_cols] = scaler.fit_transform(
    model_df[numerical_cols]
)

model_df.head()

In [ ]:
model_df.to_csv('model.csv',index=False)

<h1> EDA

<h2> CHURN DISTRIBUTION

In [ ]:
sns.countplot(x='ChurnStatus', data=merged_df)

plt.title('Customer Churn Distribution')

plt.show()

<h2> LOGIN FREQUENCY VS CHURN

In [ ]:
sns.boxplot(x='ChurnStatus',y='LoginFrequency', data=merged_df)
plt.title('Login Frequency vs Churn')

plt.show()

<h2>TOTAL SPENDING VS CHURN

In [ ]:
sns.boxplot(x='ChurnStatus',y='TotalSpent', data=merged_df)
plt.title('Total Spent vs Churn')

plt.show()

<h2> COMPLAINT COUNT VS CHURN

In [ ]:
sns.boxplot(
    x='ChurnStatus',
    y='ComplaintCount',
    data=merged_df
)
plt.title('Complaint Count vs Churn')
plt.show()


<h2> UNRESOLVED COMPLAINTS VS CHURN

In [ ]:
sns.boxplot(
    x='ChurnStatus',
    y='UnresolvedCount',
    data=merged_df
)

plt.title('Unresolved Complaints vs Churn')

plt.show()

<h2> CORRELATION HEATMAP

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(model_df.corr(),cmap='coolwarm',annot=False)
plt.title('Correlation Heatmap')

plt.show()

<h1>Model

<h2>Split the Dataset

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [ ]:
x = model_df.drop(['ChurnStatus','CustomerID','LastLoginDate'], axis=1)
y = model_df['ChurnStatus']

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=47,stratify=y)

<h2>Random Forest

In [ ]:
model_rf = RandomForestClassifier(n_estimators=100,max_depth =  10, random_state=47)
model_rf.fit(x_train,y_train)
model_rf.fit(x_train,y_train)

In [ ]:
y_pred = model_rf.predict(x_test)
y_prob = model_rf.predict_proba(x_test)[:,1]

<h1> Evaluation

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

In [ ]:
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC AUC:", roc_auc)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

<h1>Visualizations

<h2>Churn Distribution

In [ ]:
plt.figure(figsize=(10,6))

sns.countplot(x=y)
plt.title("Customer Churn Distribution")
plt.xlabel("Churn Status")
plt.ylabel("Count")
plt.show()

<h2>Confusion Matrix

In [ ]:
plt.figure(figsize=(10,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Retained', 'Churn'],
    yticklabels=['Retained', 'Churn']
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

<h2>Feature Importance

In [ ]:
feature_importance = model_rf.feature_importances_

importance_df = pd.DataFrame({'Feature': x.columns,'Importance': feature_importance})

importance_df = importance_df.sort_values(by='Importance',ascending=False)

plt.figure(figsize=(10,6))

sns.barplot(x='Importance', y='Feature',data=importance_df.head(10))

plt.title("Top 10 Important Features")

plt.show()


<h2>Model Performance

In [ ]:
metrics = {
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1 Score': f1,
    'ROC-AUC': roc_auc
}

plt.figure(figsize=(10,6))

sns.barplot(x=list(metrics.keys()), y=list(metrics.values()))

plt.ylim(0,1)

plt.title("Model Performance Metrics")
plt.ylabel("Score")
plt.show()
plt.show()


<h1>Results

<h2>Model Performance

In [ ]:
print(f"Accuracy: {metrics['Accuracy']:.2f} %")
print(f"Precision: {metrics['Precision']:.2f} %")
print(f"Recall: {metrics['Recall']:.2f} %")
print(f"F1 Score: {metrics['F1 Score']:.2f} %")
print(f"ROC AUC: {metrics['ROC-AUC']:.2f} %")


<h1>Interpretation

<ol>
<li>Accuracy shows overall prediction correctness
<li>Precision indicates how many predicted churn customers actually churned
<li>Recall indicates how many actually churned customers were correctly identified
<li>F1 score balances Precision and Recall
<li>ROC-AUC measures how effectively the model separates churn and non_churn customers

<h1>Conclusion

The Random Forest Classification model was successfully
used to predict customer churn.

The model helps identify high-risk customers and supports
business decision-making for customer retention strategies.

Further improvements can be achieved through:
- balancing the dataset
- feature engineering
- advanced ensemble models
- threshold optimization